In [ ]:
def ask_int(prompt, low, high):
    try:
        prompt = int(prompt)
    except ValueError:
        print("Incorrect value:")
        return None
    except TypeError:
        print("Type Error:")
        return None 
    
    if low <= prompt <= high:
        return prompt
    else:
        return None


In [12]:
assert ask_int("hello",0,100) == None
assert ask_int("",0,100) == None
assert ask_int("10",0,100) == 10
assert ask_int("10.5",0,100) == None
assert ask_int(True,0,100) == 1

Incorrect value:
Incorrect value:
Incorrect value:


In [ ]:
prompt = input("Enter an integer:")
result = ask_int(prompt, 0 , 100)

if result is not None:
    print(result)

Incorrect value:


In [145]:
def get_input(data, *keys):
    value = data
    for key in keys:
        if not isinstance(value,dict) or key not in value:
            return None
        else:
            value = value[key]
    return value
    


In [146]:
user = {"profile": {"role": "student"}}
assert get_input(user, "profile", "role")  ==  "student"
assert get_input(user, "skills", "python")   == None
assert get_input(user, "profile", "age")   == None 
assert get_input(user, "profile", "role", "x")  == None
assert get_input(user) == user
assert get_input(user, "profile", "role", "s") is None
# TypeError: string indices must be integers

In [166]:
import re
import math
def read_numbers(path):
    clean_data = []
    
    # exception handling file not found
    try:
        with open(path, "r",encoding="utf-8") as f:
            for line in f:
                try:
                    value = float(line)
                except ValueError:
                    continue
                if math.isfinite(value):
                    clean_data.append(value)
            
    except FileNotFoundError:
        return []
    
    # returning floating values from data
    return clean_data

In [167]:
with open("test_numbers.txt", "w", encoding="utf-8") as f:
    f.write("10\nabc\n5.5\n\n7\n3 4\n1,000\nnan\ninf\n")


read_numbers("test_numbers.txt")
# assert read_numbers("test_numbers.txt") == [10.0, 5.5, 7.0]
# assert read_numbers("does_not_exist.txt") == []
# print("all tests passed")

[10.0, 5.5, 7.0]

In [170]:
import math
def average(collection):
    
    # check if list is empty 
    if len(collection) == 0:
        return None
    
    new_collection = [value for value in collection
                      if isinstance(value,(int, float)) and not isinstance(value, bool)
                      and math.isfinite(value)]
    # check if list is empty 
    if len(new_collection) == 0:
        return None
    
    # print(new_collections)
    result = sum(new_collection) / len(new_collection)
    return result

In [172]:
print(average([2, 4, 6]))       
print(average([]))               
print(average([1, "x", None, 3]))
print(average(["a", "b"]) )      

4.0
None
2.0
None


In [185]:
# mini tool caller
import json

call_text = '{"tool":"add", "args":{"a":3, "b" : 4}}'
print(type(call_text))
print(type(json.loads(call_text)))

<class 'str'>
<class 'dict'>


In [186]:
tools = {
    "add": lambda a, b: a + b,
    "shout": lambda text: text.upper(),
    "divide": lambda a, b: a / b,
}

tools["add"]

<function __main__.<lambda>(a, b)>

In [193]:
def run_tool(call_text, tools):
    """this function inputs user query and tools and give output accordingly"""
    
    # exception handling for invalid json
    try:
        call_obj = json.loads(call_text)
    except (json.JSONDecodeError,TypeError):
        return {"ok":False , "error":"Invalid JSON"}
    
    # exception handling for input other than object(dictionary)
    if not isinstance(call_obj, dict):
        return {"ok":False , "error":"Invalid object"}
        
    tool = call_obj.get("tool",None)
    if not isinstance(tool, str):
        return {"ok":False , "error":"Missing tool name"}
        
    
    # missing tool key error
    if tool not in tools:
        return {"ok":False , "error":f"unkown tool:{tool}"}
    
    args =  call_obj.get("args",{})
    if not isinstance(args, dict):
        return {"ok":False, "error":"args must be an object"}
    
    # getting tool from tools collection
    tool_fun = tools[tool]
    # inputs = tuple([arg for arg in args.values()])
    
    # invalid, bad, zero input error handling
    try:
        result = tool_fun(**args)
    except TypeError:
        return {"ok":False , "error":"Bad argument"}
    except ZeroDivisionError:
        return {"ok":False , "error":"Cannot divide by zero"} 
    except AttributeError: 
        return {"ok":False , "error":"Attribute error"} 
    except Exception as e:
        return {"ok":False, "error":f"tool failed:{e}"}
    
    return {"ok":True , "result":result}
    
        

In [194]:
call_text = '{"tool":"divide" , "args":{ "b":7,"a":4}}'
# call_text = '[1,2]'

tools = {
    "add": lambda a, b: a + b,
    "shout": lambda text: text.upper(),
    "divide": lambda a, b: a / b,
}

result = run_tool(call_text, tools)
print(result)

{'ok': True, 'result': 0.5714285714285714}


In [195]:
assert run_tool('{"tool": ["add"], "args": {}}', tools)["ok"] is False
assert run_tool('{"tool": "hello"}', {"hello": lambda: "hi"}) == {"ok": True, "result": "hi"}
assert run_tool('{"tool": "to_int", "args": {"text": "abc"}}', {"to_int": int})["ok"] is False
r = run_tool('{"tool": "to_int", "args": {"text": "abc"}}', {"to_int": lambda text: int(text)})
json.dumps(r)     # TypeError: Object of type ValueError is not JSON serializable

'{"ok": false, "error": "tool failed:invalid literal for int() with base 10: \'abc\'"}'

In [189]:
assert run_tool('{"tool":"divide","args":{"b":1,"a":2}}', tools) == {"ok": True, "result": 2.0}
assert run_tool('{"tool":"add"}', tools)["ok"] is False                       # missing args
assert run_tool('{"tool":"add","args":{"a":1}}', tools)["ok"] is False        # missing b
assert run_tool('{"tool":"shout","args":{"text":5}}', tools)["ok"] is False   # error inside tool
assert run_tool('[1, 2]', tools)["ok"] is False
assert run_tool('{"args": {}}', tools)["ok"] is False
assert run_tool('{"tool":"power","args":{}}', tools)["ok"] is False

In [140]:
calls = {"n": 0}

def retry(task, count):
    while calls["n"] <= count:
        try:
            result = flaky()
        except ConnectionError as e:
            continue
        return result
    return "No sol after max retries"


def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("network down")
    return "ok"

assert retry(flaky, 3) == "ok"